In [ ]:
import random
from scipy.stats import ttest_ind_from_stats
import pandas as pd
from scipy.stats import chi2_contingency
from scipy.stats import fisher_exact
import math
import re

# Data Processing

## Get the A/B test data for headlines

In [ ]:
#Combine three subsets
#70% upworthy-archive-confirmatory-packages-03.12.2020 
#15% upworthy-archive-exploratory-packages-03.12.2020
#15% upworthy-archive-holdout-packages-03.12.2020
csv_file_path_1 = 'upworthy-archive-datasets/upworthy-archive-exploratory-packages-03.12.2020.csv'
csv_file_path_2 = 'upworthy-archive-datasets/upworthy-archive-confirmatory-packages-03.12.2020.csv'
csv_file_path_3 = 'upworthy-archive-datasets/upworthy-archive-holdout-packages-03.12.2020.csv'
df1 = pd.read_csv(csv_file_path_1)
df2 = pd.read_csv(csv_file_path_2)
df3 = pd.read_csv(csv_file_path_3)

df = pd.concat([df1, df2, df3])
df.to_csv('upworthy-archive-datasets/upworthy-archive-packages-all.csv', index=False)

In [ ]:
def filter_groups(group):
    return group if len(group['headline'].unique()) > 1 else None

def find_headline_pairs_with_numbers(group):
    pairs = []
    sorted_group = group.sort_values('CTR', ascending=False).reset_index()
    for i in range(len(sorted_group) - 1):
        for j in range(i + 1, len(sorted_group)):
            headline_1 = sorted_group.iloc[i]
            headline_2 = sorted_group.iloc[j]
            # Compare the CTR values and assign 1 if the first headline's CTR is higher, else 2
            higher_ctr = 0
            if headline_1['CTR'] > headline_2['CTR']:
                higher_ctr = 1
            else:
                higher_ctr = 2
            clicks = [headline_1['clicks'], headline_2['clicks']]
            non_clicks = [headline_1['impressions'] -  headline_1['clicks'], headline_2['impressions'] -  headline_2['clicks']]
            contingency_table = [clicks, non_clicks]
            #chi2, p, dof, expected = chi2_contingency(contingency_table)
            odds_ratio, p_value = fisher_exact(contingency_table)
            if headline_1['headline'] != headline_2['headline']:
                pairs.append({
                    'clickability_test_id': headline_1['clickability_test_id'],
                    'eyecatcher_id': headline_1['eyecatcher_id'],
                    'new_test_id': headline_1['new_test_id'],
                    'headline_1': headline_1['headline'],
                    'headline_2': headline_2['headline'],
                    'higher_CTR': higher_ctr,
                    'CTR_1': headline_1['CTR'],
                    'CTR_2': headline_2['CTR'],
                    'Size_1': headline_1['impressions'],
                    'Size_2': headline_2['impressions'],
                    'p_value': p_value
                })
    return pd.DataFrame(pairs)

def shuffle_and_exchange(group):
    # Shuffle the group
    shuffled_group = group.sample(frac=1).reset_index(drop=True)
    
    # Exchange the headlines with 50% probability
    for i in shuffled_group.index:
        if np.random.rand() < 0.5:
            # Swap the headlines
            shuffled_group.at[i, 'headline_1'], shuffled_group.at[i, 'headline_2'] = \
                shuffled_group.at[i, 'headline_2'], shuffled_group.at[i, 'headline_1']
            shuffled_group.at[i, 'CTR_1'], shuffled_group.at[i, 'CTR_2'] = \
                shuffled_group.at[i, 'CTR_2'], shuffled_group.at[i, 'CTR_1']
            shuffled_group.at[i, 'Size_1'], shuffled_group.at[i, 'Size_2'] = \
                shuffled_group.at[i, 'Size_2'], shuffled_group.at[i, 'Size_1']
            shuffled_group.at[i, 'higher_CTR'] = 2 if shuffled_group.at[i, 'higher_CTR'] == 1 else 1
    
    return shuffled_group


In [ ]:
df['clicks'] = df['clicks'].astype(float) 
df['impressions'] = df['impressions'].astype(float) 
df['CTR'] = df['clicks']/df['impressions']
df['created_at'] = pd.to_datetime(df['created_at'])
filtered_groups = df.groupby(['clickability_test_id', 'eyecatcher_id']).apply(filter_groups).reset_index(drop=True)
new_df = filtered_groups[['clickability_test_id', 'eyecatcher_id', 'created_at','headline', 'CTR','clicks', 'impressions']].drop_duplicates()
new_df.to_csv('upworthy-archive-datasets/ctr-all.csv', index=False)
del df, new_df

In [ ]:
df = pd.read_csv('upworthy-archive-datasets/ctr-all.csv')
df = df.sort_values(by='created_at').reset_index(drop=True)
df['new_id'] = df.groupby(['clickability_test_id', 'eyecatcher_id']).ngroup()

# Determine the indices for train and test split
train_size = int(len(df) * 0.7)
test_size = int(len(df) * 0.2)

# Create the train and test DataFrames
train_df = df.iloc[:train_size]
test_df = df.iloc[-test_size:]

# Display the resulting DataFrames
print("Train DataFrame (first 70%):")
print(train_df)

print("\nTest DataFrame (last 20%):")
print(test_df)

train_df.to_csv('upworthy-archive-datasets/train_order_by_time.csv', index=False)
test_df.to_csv('upworthy-archive-datasets/test_order_by_time.csv', index=False)

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.model_selection import train_test_split

df = pd.read_csv('upworthy-archive-datasets/ctr-all.csv')


df = df[['clickability_test_id', 'eyecatcher_id', 'headline', 'impressions', 'clicks']]
df.loc[:, 'CTR'] = df['clicks'] / df['impressions']
df['test_id'] = df.groupby(['clickability_test_id', 'eyecatcher_id']).ngroup()




train_ratio = 0.7
calibrate_ratio = 0.1
test_ratio = 0.2


gss = GroupShuffleSplit(n_splits=1, test_size=test_ratio+calibrate_ratio, random_state=42)
train_idx, temp_idx = next(gss.split(df, groups=df['test_id']))
train_df = df.iloc[train_idx]
temp_df = df.iloc[temp_idx]

gss2 = GroupShuffleSplit(n_splits=1, test_size=test_ratio/(test_ratio+calibrate_ratio), random_state=42)
calibrate_idx, test_idx = next(gss2.split(temp_df, groups=temp_df['test_id']))
calibrate_df = temp_df.iloc[calibrate_idx]
test_df = temp_df.iloc[test_idx]
del temp_df
print(len(train_df))
print(len(test_df))
print(len(calibrate_df))

In [ ]:
unique_headlines_train = set(train_df['headline'].unique())
test_df = test_df[~test_df['headline'].isin(unique_headlines_train)]
calibrate_df = calibrate_df[~calibrate_df['headline'].isin(unique_headlines_train)]
print(len(test_df))
print(len(calibrate_df))
train_df.to_csv('upworthy-archive-datasets/train.csv', index=False)
test_df.to_csv('upworthy-archive-datasets/test.csv', index=False)
calibrate_df.to_csv('upworthy-archive-datasets/calibrate.csv', index=False)

# Appendix: Get embeddings (no need to run)

In [ ]:
#We provide the code to get embeddings from OpenAI, as well as the returned embedding stored in CSV.

from openai import OpenAI
import concurrent.futures
import time
from threading import Lock
import os

client = OpenAI(api_key='XXX')

DIM = 3072
MODEL = "text-embedding-3-large"


# Assuming the DataFrame and other necessary variables are defined
# ...

class RateLimiter:
    def __init__(self, max_requests, per_seconds):
        self.max_requests = max_requests
        self.per_seconds = per_seconds
        self.lock = Lock()
        self.start_time = time.time()
        self.request_count = 0

    def wait(self):
        with self.lock:
            self.request_count += 1
            if self.request_count >= self.max_requests:
                elapsed = time.time() - self.start_time
                if elapsed < self.per_seconds:
                    time.sleep(self.per_seconds - elapsed)
                self.start_time = time.time()
                self.request_count = 0

# Initialize the rate limiter for 2000 requests per minute
rate_limiter = RateLimiter(2000, 60)

def get_embedding(text):
    rate_limiter.wait()
    try:
        response = client.embeddings.create(input=[text], model=MODEL, dimensions=DIM)
        return response.data[0].embedding
    except Exception as e:
        print(f"Error fetching embedding for text: {text}. Error: {e}")
        return None  # Return None if error

def add_embeddings(df, column_name):
    embeddings = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
        results = executor.map(get_embedding, df[column_name])
        embeddings = list(results)

    df[f'embedding_{column_name.split("_")[-1]}'] = embeddings

    
    
current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
file_path = os.path.join(parent_dir, 'upworthy-archive-datasets/ctr-all.csv')
df = pd.read_csv(file_path)
df = df[['clickability_test_id', 'eyecatcher_id', 'headline','impressions','clicks']]
df['new_test_id'] = df.groupby(['clickability_test_id', 'eyecatcher_id']).ngroup() + 1
df['clicks'] = df['clicks'].astype(float) 
df['impressions'] = df['impressions'].astype(float) 
df['CTR'] = df['clicks']/df['impressions']
    
print('getting embeddings for headline')
add_embeddings(df, 'headline')

print('save embeddings')
df.to_csv('all_test_headline_embed_3072.csv')

# Appendix: Generate pairs of headlines (no need to run)

In [ ]:
df = pd.read_csv('upworthy-archive-datasets/ctr-all.csv')
df = df[['clickability_test_id', 'eyecatcher_id', 'headline','impressions','clicks']]
#assign new test id because in the original same clickability_test_id, eyecatcher can be different
df['new_test_id'] = df.groupby(['clickability_test_id', 'eyecatcher_id']).ngroup() + 1

num_unique_new_test_id = df['new_test_id'].nunique()
print(f"Number of tests: {num_unique_new_test_id}")
print(f"Number of packages: {len(df)}")
print(f"Number of impressions: {df['impressions'].sum()}")
print(f"Number of clicks: {df['clicks'].sum()}")
df = df[['new_test_id, 'headline','impressions','clicks']]

df['clicks'] = df['clicks'].astype(float) 
df['impressions'] = df['impressions'].astype(float) 
df['CTR'] = df['clicks']/df['impressions']

headline_pairs_with_numbers_df = df.groupby(['new_test_id']).apply(find_headline_pairs_with_numbers).reset_index(drop=True)
# shuffle the data to make it balanced, which means the random guess only achieve maximum 0.5 accuracy
headline_pairs_with_numbers_df = headline_pairs_with_numbers_df.groupby(['new_test_id'], group_keys=False).apply(shuffle_and_exchange)

print('------after combining three datasets------')
print('# of original data samples = ', len(df))
print('------after choosing tests in headlines------')
print('# of tested headlines = ', len(df))
print('------after getting pairs------')
print('# of headline pairs = ', len(headline_pairs_with_numbers_df))
print('------among which, (fisher_exact test)------')
print('# of 90% significant pairs = ', len(headline_pairs_with_numbers_df[headline_pairs_with_numbers_df['p_value']<0.1]))
print('# of 95% significant pairs = ', len(headline_pairs_with_numbers_df[headline_pairs_with_numbers_df['p_value']<0.05]))
print('# of 99% significant pairs = ', len(headline_pairs_with_numbers_df[headline_pairs_with_numbers_df['p_value']<0.01]))


headline_pairs_with_numbers_df.to_csv('upworthy-archive-datasets/winner-all-new.csv', index=False)

In [ ]:
headline_pairs_with_numbers_df.head(10)